In [ ]:
cell = "GM12878"
resolution = 10000    # Hi-C resolution (bp): 10000(10kb), 25000(25kb)
display_reso = int(resolution / 1000)
root_dir = "/mnt/d/TAD-MultiOmicsNet"

In [ ]:
### Standardization of epigenetic features

In [ ]:

import pandas as pd
import numpy as np
import json
import os

TAB_PATH = f"{root_dir}/{cell}/epi_signal/epi4_GM12878_1kb.tab"
train_chrs = [f"chr{i}" for i in range(1, 13)]
val_chrs = [f"chr{i}" for i in range(13, 20)]
test_chrs = ["chr20", "chr21", "chr22", "chrX"]

df = pd.read_csv(TAB_PATH, sep="\t")
df.columns = [col.replace("'", "").strip() for col in df.columns]
coord_key = "chr" if "chr" in df.columns else "#chr"
coord_cols = [coord_key, "start", "end"]
signal_cols = [c for c in df.columns if c not in coord_cols]
signal_cols_clean = [c.replace("GM12878", "").strip("_") for c in signal_cols]
df.rename(columns=dict(zip(signal_cols, signal_cols_clean)), inplace=True)
signal_cols = signal_cols_clean
print("All columns in table:", df.columns.tolist())
print("Signal columns auto selected:", signal_cols)
print(f"Chromosome column use：{coord_key}")

# Convert to numeric values, with missing values replaced by NaN
df[signal_cols] = df[signal_cols].apply(pd.to_numeric, errors="coerce")
df = df.sort_values(by=[coord_key, "start"], ascending=True).reset_index(drop=True)

df_log = df.copy()
df_log[signal_cols] = np.log1p(df_log[signal_cols]).clip(-3, 6)

train_subset = df_log[df_log[coord_key].isin(train_chrs)]
stats = {}
for c in signal_cols:
    vals = train_subset[c].dropna()
    stats[c] = {"mu": float(vals.mean()), "sd": float(vals.std())}

df_norm = df_log.copy()
for c in signal_cols:
    mu = stats[c]["mu"]
    sd = stats[c]["sd"]
    mask_valid = ~df_norm[c].isna()
    if sd < 1e-8:
        df_norm.loc[mask_valid, c] = 0.0
    else:
        df_norm.loc[mask_valid, c] = (df_norm.loc[mask_valid, c] - mu) / sd

df_output = pd.concat([df[coord_cols], df_norm[signal_cols]], axis=1)


out_dir = f"{root_dir}/{cell}/epi_signal"

def save_chrom_tsv(chr_name, full_df, subfolder):
    save_path_dir = os.path.join(out_dir, subfolder)
    os.makedirs(save_path_dir, exist_ok=True)
    sub = full_df[full_df[coord_key] == chr_name].reset_index(drop=True)
    starts = sub["start"].values
    diffs = np.diff(starts)
    if not np.all(diffs == 1000):
        gap_pos = np.where(diffs != 1000)[0]
        print(f"WARNING: {chr_name} has a 1kb breakpoint, the first 5 breakpoint indices：{gap_pos[:5]}")
    tsv_path = os.path.join(save_path_dir, f"{chr_name}_{cell}_epi1kb.tsv")
    sub.to_csv(tsv_path, sep="\t", index=False)
    print(f"Saved {chr_name} -> {tsv_path} | bin_count={sub.shape[0]}")


for chrom in train_chrs:
    save_chrom_tsv(chrom, df_output, "all")
for chrom in val_chrs:
    save_chrom_tsv(chrom, df_output, "all")
for chrom in test_chrs:
    save_chrom_tsv(chrom, df_output, "all")

# Save the normalization parameters for reuse during inference
with open(f"{out_dir}/all/norm_stats.json", "w") as f:
    json.dump(stats, f, indent=2)
print("\n==== log1p + Z-score statistics of the training set ====")
for k, v in stats.items():
    print(f"{k:20s}: mu={v['mu']:.4f}, sd={v['sd']:.4f}")


In [ ]:
### Train set epigenetic features

In [ ]:

import pandas as pd
import numpy as np
import os

SEED_LIST = [42, 43, 44, 45, 46]
EPI_TSV = f"{root_dir}/{cell}/epi_signal/all/"
EPI_OUT = f"{root_dir}/{cell}/{display_reso}kb/samples-generate/train/"

signal_names = ["CTCF","H3k4me3","H3k27ac","H3k27me3"]
n_seg_total = 100   
seg_length = 1000

chrs_order = [f"chr{i}" for i in range(1,13)]

for SEED in SEED_LIST:
    print(f"====== SEED = {SEED} ======")
    COORD_CSV = os.path.join(EPI_OUT, f"hic_train_coords_{display_reso}kb_seed{SEED}.csv")
    OUT_NPY = os.path.join(EPI_OUT, f"epi{cell}_train_feat100_seed{SEED}.npy")
    OUT_MASK = os.path.join(EPI_OUT, f"epi{cell}_train_mask100_seed{SEED}.npy")
    
    hic_df = pd.read_csv(COORD_CSV)
    all_features = []
    all_masks = []
    
    for chrom in chrs_order:
        group_df = hic_df[hic_df["chr"] == chrom]
        print(f"Processing {chrom}, total bins: {len(group_df)}")
        epi_tsv_path = os.path.join(EPI_TSV, f"{chrom}_{cell}_epi1kb.tsv")
        if not os.path.exists(epi_tsv_path):
            print(f"!!! {epi_tsv_path} not found, skip {chrom}")
            continue
        epi_df = pd.read_csv(epi_tsv_path, sep="\t")
        epi_dict = {}
        for _, row in epi_df.iterrows():
            s = int(row["start"])
            feats = row[signal_names].values.astype(np.float32)
            epi_dict[s] = feats
    
        for _, bin_row in group_df.iterrows():
            bin_start = int(bin_row["start"])
            bin_end = int(bin_row["end"])
            center = bin_start + (bin_end - bin_start) / 2
            win_left = int(center - 50000)
    
            raw_100 = np.full((n_seg_total, len(signal_names)), np.nan)
            for seg_idx in range(n_seg_total):
                seg_l = win_left + seg_idx * seg_length
                epi_start_candidate = int(np.floor(seg_l / seg_length) * seg_length)
                if epi_start_candidate in epi_dict:
                    raw_100[seg_idx] = epi_dict[epi_start_candidate]
    
            out_mask = (~np.isnan(raw_100).all(axis=1)).astype(np.float32)
            out_feat = np.nan_to_num(raw_100, nan=0.0).astype(np.float32)
    
            all_features.append(out_feat)
            all_masks.append(out_mask)
            
    feat_arr = np.stack(all_features, axis=0)
    mask_arr = np.stack(all_masks, axis=0)
    np.save(OUT_NPY, feat_arr)
    np.save(OUT_MASK, mask_arr)
    print(f"Done! CSV total rows: {len(hic_df)}")
    print(f"feature shape: {feat_arr.shape}, mask shape:{mask_arr.shape}")

In [ ]:
### Validation set epigenetic features

In [ ]:

import pandas as pd
import numpy as np
import os

EPI_TSV = f"{root_dir}/{cell}/epi_signal/all/"
COORD_CSV = f"{root_dir}/{cell}/{display_reso}kb/samples-generate/validation/hic_val_coords13-19_{display_reso}kb.csv"
OUT_NPY = f"{root_dir}/{cell}/{display_reso}kb/samples-generate/validation/epi{cell}_validation_feat100.npy"
OUT_MASK = f"{root_dir}/{cell}/{display_reso}kb/samples-generate/validation/epi{cell}_validation_mask100.npy"

signal_names = ["CTCF","H3k4me3","H3k27ac","H3k27me3"]
n_seg_total = 100    
seg_length = 1000

# 加载Hi-C bin坐标
hic_df = pd.read_csv(COORD_CSV)
all_features = []
all_masks = []

chrs_order = [f"chr{i}" for i in range(13,20)]
for chrom in chrs_order:
    group_df = hic_df[hic_df["chr"] == chrom]
    print(f"Processing {chrom}, total bins: {len(group_df)}")
    epi_tsv_path = os.path.join(EPI_TSV, f"{chrom}_{cell}_epi1kb.tsv")
    if not os.path.exists(epi_tsv_path):
        print(f"!!! {epi_tsv_path} not found, skip {chrom}")
        continue
    epi_df = pd.read_csv(epi_tsv_path, sep="\t")
    epi_dict = {}
    for _, row in epi_df.iterrows():
        s = int(row["start"])
        feats = row[signal_names].values.astype(np.float32)
        epi_dict[s] = feats

    for _, bin_row in group_df.iterrows():
        bin_start = int(bin_row["start"])
        bin_end = int(bin_row["end"])
        center = bin_start + (bin_end - bin_start) / 2
        win_left = int(center - 50000)

        raw_100 = np.full((n_seg_total, len(signal_names)), np.nan)
        for seg_idx in range(n_seg_total):
            seg_l = win_left + seg_idx * seg_length
            epi_start_candidate = int(np.floor(seg_l / seg_length) * seg_length)
            if epi_start_candidate in epi_dict:
                raw_100[seg_idx] = epi_dict[epi_start_candidate]

        out_mask = (~np.isnan(raw_100).all(axis=1)).astype(np.float32)
        out_feat = np.nan_to_num(raw_100, nan=0.0).astype(np.float32)

        all_features.append(out_feat)
        all_masks.append(out_mask)

feat_arr = np.stack(all_features, axis=0)
mask_arr = np.stack(all_masks, axis=0)
np.save(OUT_NPY, feat_arr)
np.save(OUT_MASK, mask_arr)
print(f"Done! CSV total rows: {len(hic_df)}")
print(f"feature shape: {feat_arr.shape}, mask shape:{mask_arr.shape}")

In [ ]:
### Test set epigenetic features

In [ ]:

import pandas as pd
import numpy as np
import os

EPI_TSV = f"{root_dir}/{cell}/epi_signal/all/"  
COORD_CSV = f"{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/hic_predict_coords20-X_{display_reso}kb.csv"
OUT_NPY = f"{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/epi{cell}_predict_feat100.npy"
OUT_MASK = f"{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/epi{cell}_predict_mask100.npy"

signal_names = ["CTCF","H3k4me3","H3k27ac","H3k27me3"]
n_seg_total = 100  
seg_length = 1000

hic_df = pd.read_csv(COORD_CSV)
all_features = []
all_masks = []

chrs_order = [f"chr{i}" for i in range(20,23)]+["chrX"]
for chrom in chrs_order:
    group_df = hic_df[hic_df["chr"] == chrom]
    print(f"Processing {chrom}, total bins: {len(group_df)}")
    epi_tsv_path = os.path.join(EPI_TSV, f"{chrom}_{cell}_epi1kb.tsv")
    if not os.path.exists(epi_tsv_path):
        print(f"!!! {epi_tsv_path} not found, skip {chrom}")
        continue
    epi_df = pd.read_csv(epi_tsv_path, sep="\t")
    epi_dict = {}
    for _, row in epi_df.iterrows():
        s = int(row["start"])
        feats = row[signal_names].values.astype(np.float32)
        epi_dict[s] = feats

    for _, bin_row in group_df.iterrows():
        bin_start = int(bin_row["start"])
        bin_end = int(bin_row["end"])
        center = bin_start + (bin_end - bin_start) / 2
        win_left = center - 50000
        win_right = center + 50000

        raw_100 = np.full((n_seg_total, len(signal_names)), np.nan)
        for seg_idx in range(n_seg_total):
            seg_l = win_left + seg_idx * seg_length
            epi_start_candidate = int(np.floor(seg_l / seg_length) * seg_length)
            if epi_start_candidate in epi_dict:
                raw_100[seg_idx] = epi_dict[epi_start_candidate]

        out_mask = (~np.isnan(raw_100).all(axis=1)).astype(np.float32)
        out_feat = np.nan_to_num(raw_100, nan=0.0).astype(np.float32)

        all_features.append(out_feat)
        all_masks.append(out_mask)

feat_arr = np.stack(all_features, axis=0)
mask_arr = np.stack(all_masks, axis=0)
np.save(OUT_NPY, feat_arr)
np.save(OUT_MASK, mask_arr)
print(f"Done! CSV total rows: {len(hic_df)}")
print(f"feature shape: {feat_arr.shape}, mask shape:{mask_arr.shape}")